# 05. Fine-tuning con QLoRA

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = Path("/content/drive/MyDrive")

if IS_COLAB:
    drive = importlib.import_module("google.colab.drive")
    drive.mount("/content/drive", force_remount=False)

print(f"Entorno Colab: {IS_COLAB}")

In [ ]:
import subprocess
import sys

COLAB_PACKAGES = [
    "transformers>=4.45.0",
    "datasets==2.21.0",
    "accelerate>=0.33.0",
    "peft>=0.12.0",
    "trl>=0.10.1",
    "bitsandbytes>=0.43.1",
    "sentencepiece>=0.2.0",
    "protobuf>=5.27.2",
    "tiktoken>=0.7.0",
]

if IS_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *COLAB_PACKAGES])
    print("Dependencias de entrenamiento instaladas en Colab.")
else:
    print("Se reutiliza el entorno Python actual.")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

PROJECT_ROOT = Path("/content") if IS_COLAB else Path.cwd()
RUNTIME_ROOT = Path("/content") if IS_COLAB else PROJECT_ROOT
OUTPUT_ROOT = (RUNTIME_ROOT / "outputs/finetuned").resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT_ROOT / "data" / "processed"

# ── Configuración SFT / QLoRA ────────────────────────────────────────────────
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
TRAIN_FILE = DATA_DIR / "sft_train.jsonl"
VALID_FILE = DATA_DIR / "sft_valid.jsonl"
if not TRAIN_FILE.exists() and (PROJECT_ROOT / "sft_train.jsonl").exists():
    TRAIN_FILE = PROJECT_ROOT / "sft_train.jsonl"
if not VALID_FILE.exists() and (PROJECT_ROOT / "sft_valid.jsonl").exists():
    VALID_FILE = PROJECT_ROOT / "sft_valid.jsonl"
OUTPUT_DIR = OUTPUT_ROOT / "llama3-qlora-sft"
MERGED_OUTPUT_DIR = OUTPUT_ROOT / "llama3-qlora-sft-merged"
GGUF_FP16_PATH = OUTPUT_ROOT / "llama3.1-sft-unbiased-fp16.gguf"
GGUF_Q4_PATH = OUTPUT_ROOT / "llama3.1-sft-unbiased-q4_k_m.gguf"
OLLAMA_BUNDLE_DIR = OUTPUT_ROOT / "ollama_bundle_sft"
LLAMA_CPP_DIR = RUNTIME_ROOT / "llama.cpp"
DRIVE_EXPORT_DIR = DRIVE_ROOT / "TFM_exports/outputs/finetuned" if IS_COLAB and DRIVE_ROOT.exists() else None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MERGED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OLLAMA_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_SEED = 42
MAX_SEQ_LENGTH = 1024
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1.5e-5
WARMUP_RATIO = 0.08
LOGGING_STEPS = 5
SAVE_STEPS = 10
EVAL_STEPS = 10

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

torch.manual_seed(TRAINING_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAINING_SEED)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}")
print(f"TRAIN_FILE = {TRAIN_FILE}")
print(f"VALID_FILE = {VALID_FILE}")
print(f"Modelo base SFT = {BASE_MODEL}")
print(f"Training seed = {TRAINING_SEED}")
print(f"Artefactos persistentes en Drive: {DRIVE_EXPORT_DIR if DRIVE_EXPORT_DIR else 'deshabilitado'}")


### Exportar configuración de entrenamiento

Se guarda un resumen de hiperparámetros como artefacto persistente en
`outputs/finetuned/qlora_run_config.json`.
Este fichero se usa en el informe final para documentar el criterio de selección
de hiperparámetros.


In [ ]:
import json as _json
from pathlib import Path as _Path

_sft_train = _Path(TRAIN_FILE) if hasattr(TRAIN_FILE, '__fspath__') else TRAIN_FILE
_sft_valid = _Path(VALID_FILE) if hasattr(VALID_FILE, '__fspath__') else VALID_FILE

def _count_lines(p):
    try:
        return sum(1 for _ in open(p))
    except FileNotFoundError:
        return None

qlora_run_config = {
    "model": {
        "base_model": BASE_MODEL,
        "exported_model": "llama3.1-sft-unbiased-q4_k_m",
        "quantization": "q4_k_m (GGUF, 4-bit NF4)",
        "training_precision": "bf16 (fallback fp16 if bf16 unavailable)",
    },
    "lora": {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "bias": "none",
        "task_type": "CAUSAL_LM",
    },
    "training": {
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "optimizer": "paged_adamw_8bit",
        "lr_scheduler": "linear",
        "max_seq_length": MAX_SEQ_LENGTH,
        "gradient_checkpointing": True,
        "seed": TRAINING_SEED,
        "data_seed": TRAINING_SEED,
    },
    "dataset": {
        "train_file": str(TRAIN_FILE),
        "valid_file": str(VALID_FILE),
        "train_samples": _count_lines(TRAIN_FILE),
        "valid_samples": _count_lines(VALID_FILE),
        "format": "Alpaca-style instruction/response, completion-only training",
    },
    "hyperparameter_selection": {
        "method": "heuristic based on literature defaults and compute constraints",
        "notes": [
            "lora_r=16 and lora_alpha=32 follow common QLoRA recipes for instruction-following tasks",
            "learning_rate=1.5e-5 is conservative to avoid catastrophic forgetting",
            "batch_size=2 with gradient_accumulation=4 gives effective_batch=8 under 24 GB VRAM",
            "warmup_ratio=0.08 stabilises early training steps",
            "3 epochs balance convergence and overfitting on the small SFT dataset",
        ],
    },
}

_cfg_path = OUTPUT_ROOT / "qlora_run_config.json"
_cfg_path.parent.mkdir(parents=True, exist_ok=True)
with _cfg_path.open("w", encoding="utf-8") as _fh:
    _json.dump(qlora_run_config, _fh, ensure_ascii=False, indent=2)

print(f"Configuración QLoRA exportada → {_cfg_path}")
import pandas as _pd
_rows = []
for sec, params in qlora_run_config.items():
    if isinstance(params, dict):
        for k, v in params.items():
            _rows.append({"section": sec, "parameter": k, "value": str(v)})
_pd.set_option("display.max_colwidth", 80)
_pd.DataFrame(_rows)


In [ ]:
import importlib

import pandas as pd


def load_jsonl_dataset(path: Path) -> Dataset:
    frame = pd.read_json(path, lines=True)
    if "text" not in frame.columns:
        raise ValueError(f"El archivo {path} no contiene la columna 'text'")
    return Dataset.from_pandas(frame[["text"]], preserve_index=False)



def require_hf_token() -> str:
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        return token

    if IS_COLAB:
        try:
            userdata = importlib.import_module("google.colab.userdata")
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None

        if token:
            os.environ["HF_TOKEN"] = token
            return token

    raise RuntimeError(
        "No se encontró HF_TOKEN. En Colab, crea un secreto llamado HF_TOKEN o define la variable de entorno antes de ejecutar el entrenamiento."
    )



def build_quantization_config() -> BitsAndBytesConfig:
    compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )



def build_lora_config() -> LoraConfig:
    return LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=LORA_TARGET_MODULES,
    )



def load_model_and_tokenizer(model_name: str, token: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=token, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=token,
        quantization_config=build_quantization_config(),
        device_map="auto",
    )
    model.config.use_cache = False
    return model, tokenizer

In [ ]:
RESPONSE_TEMPLATE = "### Response\n"


def build_completion_only_collator(tokenizer):
    return DataCollatorForCompletionOnlyLM(
        response_template=RESPONSE_TEMPLATE,
        tokenizer=tokenizer,
    )

In [ ]:
MAX_PREVIEW_ROWS = 5
BASE_MODEL = BASE_MODEL

train_dataset = load_jsonl_dataset(TRAIN_FILE)
valid_dataset = load_jsonl_dataset(VALID_FILE)

dataset_report = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_dataset), "path": str(TRAIN_FILE.relative_to(PROJECT_ROOT))},
        {"split": "valid", "rows": len(valid_dataset), "path": str(VALID_FILE.relative_to(PROJECT_ROOT))},
    ]
)
display(dataset_report)
print("Si el número de filas no coincide con el run evaluado en 06_impact_evaluation.ipynb, reentrena y vuelve a exportar el modelo antes de comparar resultados.")
pd.DataFrame(train_dataset[:MAX_PREVIEW_ROWS])

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "QLoRA requiere una GPU CUDA. En Colab, activa una GPU antes de lanzar el entrenamiento."
    )

HF_TOKEN = require_hf_token()
model, tokenizer = load_model_and_tokenizer(BASE_MODEL, token=HF_TOKEN)
completion_only_collator = build_completion_only_collator(tokenizer)

total_steps = (len(train_dataset) // (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_TRAIN_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

trainer_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to=[],
    seed=TRAINING_SEED,
    data_seed=TRAINING_SEED,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=True,  # Se cambia a True para evitar que intente convertir strings a tensores
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    args=trainer_config,
    data_collator=completion_only_collator,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    peft_config=build_lora_config(),
    processing_class=tokenizer,
)

print(f"GPU activa: {torch.cuda.get_device_name(0)}")
train_output = trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(train_output)
print(f"Adaptador LoRA guardado en: {OUTPUT_DIR}")

In [ ]:
print(f"Adaptador LoRA listo en: {OUTPUT_DIR}")
print("La exportación final se ejecuta más abajo, después de fusionar y cuantizar el modelo.")

In [ ]:
import subprocess
import sys

# Actualizamos torchao a una versión compatible antes de importar peft
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16.0"])

import textwrap
from peft import PeftModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

MERGE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
MERGE_DEVICE_MAP = "auto"


def merge_adapter_to_full_model(
    base_model_name: str,
    adapter_dir: Path,
    output_dir: Path,
    token: str,
    dtype=torch.float16,
    device_map="auto",
) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        token=token,
        torch_dtype=dtype,
        device_map=device_map,
        low_cpu_mem_usage=True,
    )

    tokenizer_source = adapter_dir if (adapter_dir / "tokenizer.json").exists() else base_model_name
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, token=token, use_fast=True)

    merged_model = PeftModel.from_pretrained(base_model, adapter_dir).merge_and_unload()
    merged_model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    return output_dir


def write_ollama_modelfile(modelfile_path: Path, from_reference: str) -> Path:
    modelfile_content = f"FROM {from_reference}\n" + textwrap.dedent(
        '''\
TEMPLATE """{{- if .Prompt }}{{- if .System }}### Instruction
{{ .System }}

{{- end }}### Instruction
{{ .Prompt }}

### Response
{{- else }}{{- if .System }}### Instruction
{{ .System }}

{{- end }}{{- range .Messages }}{{- if eq .Role "user" }}### Instruction
{{ .Content }}

### Response
{{- else if eq .Role "assistant" }}{{ .Content }}

{{- end }}{{- end }}{{- end }}"""
PARAMETER stop "### Instruction"
'''
    )
    modelfile_path.write_text(modelfile_content, encoding="utf-8")
    return modelfile_path


HF_TOKEN = require_hf_token()
merged_dir = merge_adapter_to_full_model(
    base_model_name=BASE_MODEL,
    adapter_dir=OUTPUT_DIR,
    output_dir=MERGED_OUTPUT_DIR,
    token=HF_TOKEN,
    dtype=MERGE_DTYPE,
    device_map=MERGE_DEVICE_MAP,
)
merged_modelfile = write_ollama_modelfile(merged_dir / "Modelfile", ".")
print(f"Modelo fusionado guardado en: {merged_dir}")
print(f"Modelfile HF/Ollama generado en: {merged_modelfile}")

### 1. Preparar llama.cpp
Clonamos el repositorio de llama.cpp e instalamos las dependencias necesarias para la conversión a GGUF.

In [ ]:
import subprocess

if not LLAMA_CPP_DIR.exists():
    subprocess.check_call([
        "git",
        "clone",
        "https://github.com/ggerganov/llama.cpp.git",
        str(LLAMA_CPP_DIR),
    ])

convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
if not convert_script.exists():
    raise FileNotFoundError(f"No se encontró el script de conversión en {convert_script}")

print(f"llama.cpp listo en: {LLAMA_CPP_DIR}")

### 2. Convertir y Cuantizar
Convertimos el modelo de Hugging Face a GGUF (FP16) y luego lo cuantizamos a `q4_k_m` (aprox. 4.8GB - 5GB).

In [ ]:
import subprocess

convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
quantize_binary = LLAMA_CPP_DIR / "build/bin/llama-quantize"
build_dir = LLAMA_CPP_DIR / "build"

subprocess.check_call([
    "cmake",
    "-S",
    str(LLAMA_CPP_DIR),
    "-B",
    str(build_dir),
])
subprocess.check_call([
    "cmake",
    "--build",
    str(build_dir),
    "--config",
    "Release",
    "-t",
    "llama-quantize",
])

if GGUF_FP16_PATH.exists():
    GGUF_FP16_PATH.unlink()
if GGUF_Q4_PATH.exists():
    GGUF_Q4_PATH.unlink()

subprocess.check_call([
    sys.executable,
    str(convert_script),
    str(MERGED_OUTPUT_DIR),
    "--outfile",
    str(GGUF_FP16_PATH),
    "--outtype",
    "f16",
])
subprocess.check_call([
    str(quantize_binary),
    str(GGUF_FP16_PATH),
    str(GGUF_Q4_PATH),
    "q4_k_m",
])

if GGUF_FP16_PATH.exists():
    GGUF_FP16_PATH.unlink()
    print("Modelo FP16 temporal eliminado para ahorrar espacio.")

print(f"Cuantización completada. GGUF final en: {GGUF_Q4_PATH}")

### 3. Crear el Modelfile para Ollama
Generamos un nuevo Modelfile que apunte directamente a nuestro modelo cuantizado en formato GGUF.

In [ ]:
import shutil


def sync_artifact_to_drive(source: Path, destination_dir: Path) -> Path:
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / source.name
    if source.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)
    else:
        shutil.copy2(source, destination)
    return destination


bundle_gguf_path = OLLAMA_BUNDLE_DIR / GGUF_Q4_PATH.name
shutil.copy2(GGUF_Q4_PATH, bundle_gguf_path)
bundle_modelfile = write_ollama_modelfile(OLLAMA_BUNDLE_DIR / "Modelfile", f"./{bundle_gguf_path.name}")

print(f"Bundle Ollama listo en: {OLLAMA_BUNDLE_DIR}")
print(f"Modelfile final: {bundle_modelfile}")

if DRIVE_EXPORT_DIR is not None:
    sync_artifact_to_drive(OUTPUT_DIR, DRIVE_EXPORT_DIR)
    sync_artifact_to_drive(MERGED_OUTPUT_DIR, DRIVE_EXPORT_DIR)
    drive_bundle_path = sync_artifact_to_drive(OLLAMA_BUNDLE_DIR, DRIVE_EXPORT_DIR)
    print(f"Bundle sincronizado en Drive: {drive_bundle_path}")

In [ ]:
bundle_zip_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_ROOT / "ollama_bundle"),
        format="zip",
        root_dir=str(OLLAMA_BUNDLE_DIR),
    )
)
print(f"Bundle comprimido en: {bundle_zip_path}")
print(f"GGUF final disponible en: {GGUF_Q4_PATH}")